# Libraries

In [42]:
# Stat Libs
import pandas as pd
from pandas.api.types import CategoricalDtype
import pickle
import re

# Stat Libs
import statsmodels.api as sm
from itertools import product
from functools import reduce

# Statistical libs
from sklearn.preprocessing import MultiLabelBinarizer


# Load Data

In [43]:
df1 = pd.read_pickle(r".\df_unmerged\df1_unmerged.pkl")
df2 = pd.read_pickle(r".\df_unmerged\df2_unmerged.pkl")
df3 = pd.read_pickle(r".\df_unmerged\df3_unmerged.pkl")
df4 = pd.read_pickle(r".\df_unmerged\df4_unmerged.pkl")

In [44]:
dfis = [df1, df2, df3, df4]
iss = [1, 2, 3, 4]

# Dummies
"_List” data: List element data, were encoded through dummy creation. The problem was that list-element data cannot be used from models. Firstly, list-element rows were exploded to one row per element of list. However, this inflates sample size, as rows increase and duplicate for same nct_id if more than one element occur in a list-row. For this reason, these data were then grouped by nct_id, so number of rows remained the same as initial datasets (df0, df1, df2, df3, df4, df5).

In [45]:
## Dummies
def fun_dum_enc(dfi, cols):
    for col in cols:  
        df_expl = dfi.copy()
        df_expl = df_expl.explode(col)

        df_expl[col] = df_expl[col].astype('category') 
        df_expl[col] = df_expl[col].cat.remove_unused_categories()
        df_expl[col] = df_expl[col].astype('str') #str cause of error in encoding. After astype(cat) so to drop unused categories

        dummies = pd.get_dummies(df_expl[col], drop_first = False, dtype = int, prefix = col , prefix_sep='_')
        
        dummies.index = df_expl.index # ensure same indexing with df_expl
        dummies = dummies.groupby(dummies.index).sum().clip(upper = 1) 
        # clip: if a row has double entry data ['UNSPES', 'UNSPES'] it avoids double vounting with sum().

        dfi = pd.concat([dfi.drop(columns = [col], axis = 1), dummies], axis = 1)  
    return dfi

### Cols
def fun_dum_cols(dfis):  # In case they are not the same.
    dum_cols = []
    for dfi in dfis: # loop inputed in case dfis have not all the same columns. # * Plus not to run function into funtion.
        dum_cols = dum_cols + [[col for col in dfi.columns if '_List' in col]]
    return dum_cols

dum_cols = fun_dum_cols(dfis) 

# Apply
# * loop so not to run function into function
df1 = fun_dum_enc(df1, dum_cols[0])
df2 = fun_dum_enc(df2, dum_cols[1])
df3 = fun_dum_enc(df3, dum_cols[2])
df4 = fun_dum_enc(df4, dum_cols[3])


# Example
display(dum_cols[0])
df1[[col for col in df1.columns if '_list' in col.lower()]] #.head()  # Transposed for better view

['Sex_List',
 'Age_List',
 'Funder_Industry_List',
 'Intervention_Type_List',
 'Intervention_Route_List',
 'Conditions_Detail_List',
 'Adverse_List',
 'Adverse_System_List',
 'Intervention_Model_List',
 'Masking_List',
 'Masking_Detail_List',
 'Primary_Purpose_List',
 'Continents_List']

,Sex_List_ALL,Sex_List_FEMALE,Sex_List_MALE,Age_List_ADULT,Age_List_CHILD,Age_List_OLDER_ADULT,Funder_Industry_List_FED,Funder_Industry_List_INDIV,Funder_Industry_List_INDUSTRY,Funder_Industry_List_NETWORK,...,Primary_Purpose_List_SCREENING,Primary_Purpose_List_SUPPORTIVE_CARE,Primary_Purpose_List_TREATMENT,Continents_List_Africa,Continents_List_Asia,Continents_List_Cont_Other,Continents_List_Europe,Continents_List_North America,Continents_List_Oceania,Continents_List_South America
0,0,0,1,1,0,0,0,0,1,0,...,0,0,1,0,1,0,0,0,0,0
1,0,0,1,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,1,0,0,1,0,1,0,0,1,0,...,0,0,1,0,0,0,0,1,0,0
3,1,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
4,1,0,0,1,0,1,0,0,1,0,...,0,0,1,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21824,1,0,0,1,1,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
21825,1,0,0,1,0,1,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
21826,1,0,0,1,0,0,0,0,1,0,...,0,0,1,0,1,0,0,0,0,0
21827,1,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0


In [46]:
dfis = [df1, df2, df3, df4]
iss = [1, 2, 3, 4]

- Drop first was not done through get_dummies command, so to choose the column to drop, based on its characterisic. e.g., drop phase 0 is preferred that to drop phase 3.
- This file provides dta for Sparsity. Will not dropp any columns. 
- This way see all levels is achiebed. Also, the interactions is loaded from Dummies file not this. 
So, not dropping first does not alter interactions sparsity analysis.

# Binary

In [47]:
# Binary Encoding
def fun_bin_enc(dfi, cols):
    dfi = dfi.copy()
    for col in cols:
        cats = CategoricalDtype(categories = sorted(dfi[col].dropna().unique()), ordered = False)
        dfi[col] = dfi[col].astype(cats).cat.codes
    return dfi


### Cols
def fun_bin_cols(dfis):
    bin_cols = []
    for dfi in dfis:
        bin_cols = bin_cols + [[col for col in dfi.columns if '_Categ' in col.lower() or '_Bin' in col]] 
    return bin_cols

# Apply
bin_cols = fun_bin_cols(dfis)

df1 = fun_bin_enc(df1, bin_cols[0])
df2 = fun_bin_enc(df2, bin_cols[1])
df3 = fun_bin_enc(df3, bin_cols[2])
df4 = fun_bin_enc(df4, bin_cols[3])

# Example
display(df1['Study_Status_Bin'].value_counts())  # Completed = 0, Terminated = 1
display(bin_cols[0])  # bin_cols[0] --> bin_cols of df1
display(df1[bin_cols[0]])

Study_Status_Bin
0    18646
1     3183
Name: count, dtype: int64

['Study_Status_Bin',
 'Placebo_Bin',
 'Standard_Care_Bin',
 'Healthy_Bin',
 'Covid_19_Bin',
 'Adverse_Bin',
 'Allocation_Bin',
 'Global_Bin']

,Study_Status_Bin,Placebo_Bin,Standard_Care_Bin,Healthy_Bin,Covid_19_Bin,Adverse_Bin,Allocation_Bin,Global_Bin
0,0,0,0,1,0,0,2,0
1,0,1,0,1,0,0,2,0
2,0,0,0,0,0,0,1,0
3,0,0,0,1,0,0,2,0
4,1,1,1,0,0,0,2,0
...,...,...,...,...,...,...,...,...
21824,0,0,0,0,0,0,0,0
21825,0,0,0,1,0,1,0,0
21826,0,0,0,1,0,0,2,0
21827,0,1,0,0,0,0,2,0


In [48]:
dfis = [df1, df2, df3, df4]
iss = [1, 2, 3, 4]

# Save dfis

In [49]:
df1.to_pickle(r".\df_dummies_unmerged\df1_dummies_unmerged.pkl")
df2.to_pickle(r".\df_dummies_unmerged\df2_dummies_unmerged.pkl")
df3.to_pickle(r".\df_dummies_unmerged\df3_dummies_unmerged.pkl")
df4.to_pickle(r".\df_dummies_unmerged\df4_dummies_unmerged.pkl")